# 03 — Optimization: Gradient Descent and SGD

**CS231n Lecture 3 (second half).** Notebook 02 gave you a loss and a gradient.
Neither does anything on its own. This notebook is about the search procedure
that turns them into a trained model, and about the one hyperparameter that
matters more than all the others combined.

| You write | Why |
|---|---|
| `predict_linear` | trivial, but it anchors the shapes |
| `train_linear` | the minibatch SGD loop — three lines that are the whole field |
| `loss_along_direction` | slice the loss surface so you can *see* it |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (get_cifar10, CLASSES, rel_error, grade, show_solution,
                          use, plot_series, plot_weight_templates,
                          eval_numerical_gradient, setup_plots, SERIES)
from solutions.nb02 import svm_loss_vectorized, softmax_loss_vectorized
from solutions.nb03 import make_batches

setup_plots()
np.random.seed(0)

data = get_cifar10(num_train=5000, num_val=500, num_test=500,
                   flatten=True, subtract_mean=True)
add_bias = lambda X: np.hstack([X, np.ones((X.shape[0], 1))])
X_train = add_bias(data['X_train']); y_train = data['y_train']
X_val   = add_bias(data['X_val']);   y_val   = data['y_val']
X_test  = add_bias(data['X_test']);  y_test  = data['y_test']
D, C = X_train.shape[1], 10
print("X_train", X_train.shape)

## 1. Three strategies, in order of how bad they are

### Strategy 1: random search

Sample random `W`, keep the best. It is a real baseline and it is worth running
once so you feel how hopeless it is in 3073 dimensions.

In [ ]:
best_loss, best_W = float('inf'), None
t0 = time.time()
for i in range(200):
    W_try = np.random.randn(D, C) * 1e-4
    loss, _ = svm_loss_vectorized(W_try, X_train, y_train, 0.0)
    if loss < best_loss:
        best_loss, best_W = loss, W_try
acc = np.mean(np.argmax(X_val.dot(best_W), axis=1) == y_val)
print("random search: 200 samples in %.1f s -> best loss %.3f, val acc %.3f"
      % (time.time() - t0, best_loss, acc))

#~15% — barely above chance. The search space is far too big to sample.

### Strategy 2: random local search

Start somewhere, jump a small random step, keep the step only if the loss went
down. Better — this actually descends — but every step costs a full loss
evaluation and most steps are rejected.

In [ ]:
W = np.random.randn(D, C) * 1e-3
cur_loss, _ = svm_loss_vectorized(W, X_train, y_train, 0.0)
accepted = 0
t0 = time.time()
for i in range(200):
    step = np.random.randn(D, C) * 1e-4
    loss, _ = svm_loss_vectorized(W + step, X_train, y_train, 0.0)
    if loss < cur_loss:
        W, cur_loss = W + step, loss
        accepted += 1
print("random local search: %d/200 steps accepted in %.1f s -> loss %.3f, val acc %.3f"
      % (accepted, time.time() - t0, cur_loss,
         np.mean(np.argmax(X_val.dot(W), axis=1) == y_val)))

### Strategy 3: follow the gradient

The gradient is the direction of steepest *ascent*, so `-gradient` is the single
best direction to step in. You get it in **one** loss evaluation instead of
guessing. That is the entire difference, and it is enough.

## 2. Numeric vs analytic gradients

Two ways to get $\nabla_W L$:

| | Numeric | Analytic |
|---|---|---|
| how | finite differences | calculus |
| cost | 2 evaluations **per parameter** | 1 backward pass, all parameters |
| exact? | no, $O(h^2)$ error and kink problems | yes |
| easy to get wrong? | almost impossible | very easy |

For a 3073 x 10 `W` that is 61,460 loss evaluations for one numeric gradient.
Completely unusable for training — but perfect for **checking**, which is the
only thing it is for.

Use the **centred** difference. The one-sided version $\frac{f(x+h)-f(x)}{h}$
has error $O(h)$; the centred one $\frac{f(x+h)-f(x-h)}{2h}$ has error $O(h^2)$
for twice the cost. Always worth it.

In [ ]:
# Watch the error scale with h, and watch it stop improving.
W_small = np.random.randn(20, 4) * 0.1
# 20 features and 4 classes, so labels must be remapped into range
Xs, ys = X_train[:50, :20], y_train[:50] % 4
_, analytic = softmax_loss_vectorized(W_small, Xs, ys, 0.0)
f = lambda w: softmax_loss_vectorized(w, Xs, ys, 0.0)[0]

print("  h        one-sided      centred")
for h in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]:
    g_one = np.zeros_like(W_small)
    g_cen = np.zeros_like(W_small)
    for idx in np.ndindex(W_small.shape):
        old = W_small[idx]
        W_small[idx] = old + h; fp = f(W_small)
        W_small[idx] = old - h; fm = f(W_small)
        W_small[idx] = old;     f0 = f(W_small)
        g_one[idx] = (fp - f0) / h
        g_cen[idx] = (fp - fm) / (2 * h)
    print("  %-8.0e %.3e    %.3e"
          % (h, rel_error(g_one, analytic), rel_error(g_cen, analytic)))

**Read the bottom rows.** The error improves as `h` shrinks — and then gets
*worse* again. Two errors fight each other:

- **truncation error** shrinks as `h` shrinks (the Taylor remainder)
- **round-off error** grows as `h` shrinks, because `f(x+h) - f(x-h)` is the
  difference of two nearly-equal float64 numbers, and you lose significant
  digits to cancellation

The sweet spot for float64 is around `h = 1e-5`. This is why `1e-5` is
hard-coded in every gradient checker including the one in `cs231n_utils`, and
why "just use a smaller h" is the wrong instinct.

> **Check in float64, always.** In float32 the round-off floor is so high that a
> gradient check is meaningless. If your model is float32, cast to float64 for
> the check.

## 3. TODO 1 — `predict_linear`

Warm-up. One line.

In [ ]:
def predict_linear(W, X):
    """X (N, D), W (D, C) -> (N,) predicted class indices."""
    pass  # TODO

In [ ]:
grade('nb03', 'predict_linear', predict_linear)

## 4. TODO 2 — the SGD training loop

**Why minibatches.** The true gradient averages over all N examples. With
N = 1.2M (ImageNet) that is absurd per step. But the training data is redundant
— a batch of 256 gives a *noisy estimate of the same direction* for 0.02% of the
cost, so you can take 5000 noisy steps instead of 1 exact one. Noisy steps win,
easily.

Batch size is mostly a hardware decision (largest that fits in memory) and
usually a power of 2. It is not a hyperparameter you agonise over — unlike the
learning rate.

Note the signature: `batches` is a **precomputed list of index arrays**. That
keeps the loop deterministic and separates "which examples" from "what to do
with them", which is the part you're actually learning.

In [ ]:
def train_linear(loss_fn, W0, X, y, batches, learning_rate, reg):
    """Vanilla minibatch SGD. Returns (W, loss_history).

    For each index array `idx` in `batches`:
      1. evaluate loss_fn on X[idx], y[idx]
      2. step W against the gradient
      3. record the loss
    """
    W = W0.copy()
    loss_history = []
    # TODO
    return W, loss_history

In [ ]:
grade('nb03', 'train_linear', train_linear)

In [ ]:
train = use('nb03', 'train_linear', train_linear)
predict = use('nb03', 'predict_linear', predict_linear)

# Init small enough that the starting scores are near zero -- then the initial
# loss should be -log(1/10) = 2.303, the sanity check from notebook 02.
W0 = np.random.RandomState(0).randn(D, C) * 1e-5
print("loss at init: %.4f   (expected %.4f)"
      % (softmax_loss_vectorized(W0, X_train, y_train, 0.0)[0], -np.log(0.1)))

batches = make_batches(len(X_train), 200, 1500, seed=0)

t0 = time.time()
W, hist = train(softmax_loss_vectorized, W0, X_train, y_train, batches, 5e-6, 1e3)
print("%.1f s   train %.3f   val %.3f"
      % (time.time() - t0,
         np.mean(predict(W, X_train) == y_train),
         np.mean(predict(W, X_val) == y_val)))

## 5. The learning rate — the hyperparameter that matters

If you tune exactly one thing, tune this. The four regimes below are worth being
able to recognise from the shape of the loss curve alone, because that is how
you will actually diagnose training runs.

In [ ]:
lrs = [1e-9, 5e-6, 1e-4, 1e-3]
labels = ['1e-9  (too low)', '5e-6  (about right)',
          '1e-4  (too high)', '1e-3  (diverges)']
curves = []
for lr, lab in zip(lrs, labels):
    _, h = train(softmax_loss_vectorized, W0, X_train, y_train, batches, lr, 0.0)
    h = np.asarray(h)
    curves.append((lab, h))
    blew_up = not np.all(np.isfinite(h))
    print("lr %8.0e  first %7.3f  mid %10.3f  last %12.3f  %s"
          % (lr, h[0], h[len(h) // 2], h[-1], "NON-FINITE" if blew_up else ""))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for i, (lab, h) in enumerate(curves):
    # replace inf/nan so the diverging curve still draws up to where it died
    hp = np.where(np.isfinite(h), h, np.nan)
    axes[0].plot(np.arange(len(hp)), hp, color=SERIES[i], label=lab)
    axes[1].plot(np.arange(len(hp)), hp, color=SERIES[i], label=lab)
axes[0].set_ylim(0, 6)
axes[0].set_title("Loss vs learning rate (clipped to [0, 6])")
axes[1].set_yscale('log')
axes[1].set_title("same, log scale")
for ax in axes:
    ax.set_xlabel("iteration"); ax.set_ylabel("loss")
    ax.legend(); ax.set_axisbelow(True)
plt.show()

### The four shapes

Every curve starts at 2.303. Where it goes from there is the diagnosis:

| Shape | Meaning | Fix |
|---|---|---|
| almost flat (`1e-9`) | LR far too **low** — it is learning, just not in your lifetime | raise it 10x |
| smooth, decreasing, plateaus (`5e-6`) | about right | maybe decay it later |
| drops then climbs back up (`1e-4`) | LR too **high** — it is overshooting the minimum every step and bouncing up the walls | lower it 3-10x |
| `inf` / `nan` (`1e-3`) | divergence — each overshoot is bigger than the last | lower it 100x |

The third row is the one people misread. A loss that goes *up* is not a broken
gradient; the gradient direction is correct and the step along it is simply too
long, so you land further up the opposite side of the valley than where you
started. Notice it still starts by dropping — high learning rates often look
fine for the first few dozen iterations.

Two working habits from this:

- **Tune the learning rate on a log scale.** `1e-7, 1e-6, 1e-5` — never
  `1e-6, 2e-6, 3e-6`. The interesting variation is in the order of magnitude.
- **A loss of `nan` is almost always the learning rate.** It is the first thing
  to check, before you go looking for a bug in your gradient.

## 6. TODO 3 — see the loss surface

`W` lives in 30,730 dimensions, so you cannot plot the loss surface. What you
*can* do is take a 1D or 2D slice: pick a random direction `d`, and plot
$L(W + \alpha d)$ as $\alpha$ varies.

In [ ]:
def loss_along_direction(loss_fn, W, direction, alphas, X, y, reg):
    """Return an array of loss_fn(W + a * direction) for each a in alphas."""
    out = np.zeros(len(alphas))
    # TODO
    return out

In [ ]:
grade('nb03', 'loss_along_direction', loss_along_direction)

In [ ]:
slice_ = use('nb03', 'loss_along_direction', loss_along_direction)

Wc = W.copy()
alphas = np.linspace(-3, 3, 61)
Xb, yb = X_train[:500], y_train[:500]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for i in range(3):
    d = np.random.randn(*Wc.shape)
    d /= np.linalg.norm(d)
    axes[0].plot(alphas, slice_(svm_loss_vectorized, Wc, d, alphas, Xb, yb, 0.0),
                 color=SERIES[i], label="direction %d" % (i + 1))
    axes[1].plot(alphas, slice_(softmax_loss_vectorized, Wc, d, alphas, Xb, yb, 0.0),
                 color=SERIES[i], label="direction %d" % (i + 1))
axes[0].set_title("SVM loss along random directions")
axes[1].set_title("Softmax loss along random directions")
for ax in axes:
    ax.set_xlabel("step size along direction")
    ax.set_ylabel("loss")
    ax.legend()
    ax.set_axisbelow(True)
plt.show()

### What you're looking at

**The SVM slices are piecewise linear with visible corners.** That is the hinge:
each training example contributes a `max(0, ...)` that switches on or off at
some $\alpha$, and each switch is a kink. Sum many kinked functions and you get
a bowl made of flat facets.

This is also the honest answer to "is the loss convex?" — for a **linear**
classifier, yes, it is convex, and those corners are where it is
non-differentiable rather than where it is non-convex. Once you stack a hidden
layer in notebook 05, convexity is gone for good and the corners are the least
of your problems.

**The softmax slices are smooth**, because $\log \sum e^{s}$ is smooth
everywhere. This is exactly why softmax gradient-checks cleanly and SVM does not.

In [ ]:
# A 2D slice: the loss on a plane through W. Coarse grid, this is O(n^2) evals.
from solutions.nb03 import loss_surface_2d

d1 = np.random.randn(*Wc.shape); d1 /= np.linalg.norm(d1)
d2 = np.random.randn(*Wc.shape); d2 /= np.linalg.norm(d2)
g = np.linspace(-2.5, 2.5, 25)
Z = loss_surface_2d(svm_loss_vectorized, Wc, d1, d2, g, g, X_train[:300], y_train[:300], 0.0)

fig, ax = plt.subplots(figsize=(4.6, 3.8))
cs = ax.contourf(g, g, Z, levels=18, cmap='Blues_r')
ax.contour(g, g, Z, levels=18, colors='white', linewidths=0.4, alpha=0.5)
ax.plot(0, 0, marker='o', color='#eb6834', markersize=7, zorder=5)
ax.annotate("current W", xy=(0, 0), xytext=(6, 6), textcoords='offset points',
            fontsize=8, color='#eb6834', weight='semibold')
ax.set_title("SVM loss on a 2D slice")
ax.set_xlabel("direction 1"); ax.set_ylabel("direction 2")
ax.grid(False)
fig.colorbar(cs, ax=ax, label='loss')
plt.show()

A single-hue sequential ramp, light-to-dark, because this encodes **magnitude**.
A rainbow colormap here would invent boundaries the data does not have — a ridge
appears wherever the hue happens to change fast. Same rule as any other chart:
sequential data, one hue.

## 7. Putting it together — a real hyperparameter search

Random search over a log-uniform grid, scored on validation, test touched once.
This is the workflow for every model you will train.

> **Random search beats grid search** when some hyperparameters matter much more
> than others — and they always do. With a grid, 9 runs over `lr` x `reg` only
> ever try 3 distinct learning rates. With random sampling, 9 runs try 9. You
> get more resolution on the dimension that matters, for free.

In [ ]:
rng = np.random.RandomState(1)
results, best = {}, (0.0, None, None)

t0 = time.time()
for trial in range(12):
    lr  = 10 ** rng.uniform(-7.0, -5.0)
    reg = 10 ** rng.uniform(2.0, 4.5)
    b = make_batches(len(X_train), 200, 700, seed=trial)
    Wt, _ = train(softmax_loss_vectorized, W0, X_train, y_train, b, lr, reg)
    tr = np.mean(predict(Wt, X_train) == y_train)
    va = np.mean(predict(Wt, X_val) == y_val)
    results[(lr, reg)] = (tr, va)
    if va > best[0]:
        best = (va, Wt, (lr, reg))
    print("lr %.2e  reg %.2e  train %.3f  val %.3f" % (lr, reg, tr, va))

print("\n%.1f s | best val %.3f at lr %.2e reg %.2e"
      % (time.time() - t0, best[0], best[2][0], best[2][1]))

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.8))
lrs_ = [k[0] for k in results]
regs_ = [k[1] for k in results]
vals = [v[1] for v in results.values()]
sc = ax.scatter(lrs_, regs_, c=vals, s=110, cmap='Blues', edgecolor='#52514e',
                linewidth=0.6, vmin=min(vals), vmax=max(vals))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel("learning rate"); ax.set_ylabel("regularisation")
ax.set_title("Random search — validation accuracy")
ax.set_axisbelow(True)
fig.colorbar(sc, ax=ax, label='val accuracy')
for (lr, reg), (tr, va) in results.items():
    ax.annotate("%.2f" % va, xy=(lr, reg), xytext=(0, -14),
                textcoords='offset points', ha='center', fontsize=7,
                color='#52514e')
plt.show()

Every point is labelled with its value, so the encoding never rests on colour
alone. If the best points cluster at the edge of your sampled range, **widen the
range and search again** — you have not found the optimum, you have found the
boundary of where you looked.

In [ ]:
W_best = best[1]
print("FINAL test accuracy: %.3f" % np.mean(predict(W_best, X_test) == y_test))
plot_weight_templates(W_best, CLASSES, title="Best model — learned templates")
plt.show()

## 8. Recap

1. Why is random search hopeless in 3073 dimensions but useful for 2 hyperparameters?
2. Numeric gradients are nearly impossible to get wrong and useless for training.
   Analytic gradients are the reverse. What is the practice that uses both?
3. Why does gradient-check error get *worse* below about `h = 1e-6`?
4. Why does `h = 1e-5` appear hard-coded everywhere?
5. Minibatch SGD uses a *wrong* gradient at every step and beats full-batch
   descent. Why?
6. Name the four learning-rate curve shapes and the fix for each.
7. Why sample learning rates log-uniformly rather than uniformly?
8. Why are the SVM loss slices kinked and the softmax slices smooth?
9. Your best hyperparameters sit at the edge of the sampled range. What now?

In [ ]:
# show_solution('nb03', 'train_linear')
# show_solution('nb03', 'loss_along_direction')